# Sport Logiq CSV EDA on Advanced Metric Queries

In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

In [19]:

# define teams in each conference
AHA = [
    'Air Force Academy Falcons', 
    'Army Black Knights',
    'Bentley University Falcons',
    'Canisius College Golden Griffins',
    'College of the Holy Cross Crusaders',
    'Mercyhurst University Lakers',
    'Niagara University Purple Eagles',
    'Robert Morris University Colonials',
    'Rochester Institute of Technology Tigers',
    'Sacred Heart University Pioneers'
]

CCHA = [
    'Augustana University Vikings',
    'Bemidji State University Beavers',
    'Bowling Green State University Falcons',
    'Ferris State University Bulldogs',
    'Lake Superior State University Lakers',
    'Michigan Tech University Huskies',
    'Minnesota State University Mavericks',
    'Northern Michigan University Wildcats',
    'The University of St. Thomas Tommies'
]

ECAC = [
    'Brown University Bears',
    'Clarkson University Golden Knights',
    'Colgate University Raiders',
    'Cornell University Big Red',
    'Dartmouth College Big Green',
    'Harvard University Crimson',
    'Princeton University Tigers',
    'Quinnipiac University Bobcats',
    'Rensselaer Polytechnic Institute Engineers',
    'St. Lawrence University Saints',
    'Union College Garnet Chargers',
    'Yale University Bulldogs'
]

HE = [
    'Boston College Eagles',
    'Boston University Terriers',
    'Merrimack College Warriors',
    'Northeastern University Huskies',
    'UMass Lowell River Hawks',
    'UMass Minutemen',
    'University of Connecticut Huskies',
    'University of Maine Black Bears',
    'University of New Hampshire Wildcats',
    'University of Vermont Catamounts',
    'Providence College Friars'
]

NCHC = [
    'Arizona State University Sun Devils',
    'Colorado College Tigers',
    'University of Denver Pioneers',
    'Miami University (Ohio) RedHawks',
    'University of Minnesota-Duluth Bulldogs',
    'University of North Dakota Fighting Hawks',
    'University of Nebraska Omaha Mavericks',
    'St. Cloud State University Huskies',
    'Western Michigan University Broncos'
]

BT = [
    'Michigan State University Spartans',
    'Michigan Wolverines',
    'Ohio State University Buckeyes',
    'Penn State University Nittany Lions'
    'University Of Notre Dame Fighting Irish',
    'University of Minnesota Golden Gophers',
    'University of Wisconsin Badgers'
]

INDE = [
    'University of Alaska Anchorage Seawolves',
    'University of Alaska Fairbanks Nanooks',
    'Lindenwood University Lions',
    'Long Island University Sharks',
    'Stonehill College Skyhawks'
]

In [20]:

import glob
file_paths = glob.glob('/Users/longer/uconn/team_data/*.csv')
dataframes = [pd.read_csv(file, index_col='Team') for file in file_paths]

master_df = pd.concat(dataframes, axis=1)
master_df = master_df.loc[:, ~master_df.columns.duplicated()]
master_df = master_df.reset_index()
master_df.to_csv("Master_Team_Baselines.csv", index=False)


ValueError: Index Team invalid

In [21]:
print(master_df.columns)

Index(['Team', 'Successful OZ Open Ice Dekes', 'Failed OZ Open Ice Dekes',
       'OZ Open Ice Deke Success Rate', 'Successful DZ Open Ice Dekes',
       'Failed DZ Open Ice Dekes', 'DZ Open Ice Deke Success Rate',
       'Successful NZ Open Ice Dekes', 'Failed NZ Open Ice Dekes',
       'NZ Open Ice Deke Success Rate',
       ...
       'Total DZ Penalties', 'Total NZ Undisciplined Penalties',
       'Total NZ Obstruction Penalties', 'Total NZ Penalties',
       'Total Undisciplined Penalties', 'Total Obstruction Penalties',
       'Total Penalties', 'Unnamed: 0', 'Conference', 'conference'],
      dtype='object', length=390)


In [22]:
conference_mapping = {}
lists = [AHA, CCHA, ECAC, HE, NCHC, BT, INDE]
names = ['Atlantic Hockey', 'CCHA', 'ECAC', 'Hockey East', 'NCHC', 'Big Ten', 'Independent']
for conf_list, conf_name in zip(lists, names):
    for team in conf_list:
        conference_mapping[team] = conf_name



In [23]:
master_df['conference'] = master_df['Team'].map(conference_mapping)
teams = master_df['Team']
conferences = master_df['conference']
numeric_metrics = master_df.drop(columns=['Team', 'conference', 'Conference'])


In [24]:

master_df.head()


,Team,Successful OZ Open Ice Dekes,Failed OZ Open Ice Dekes,OZ Open Ice Deke Success Rate,Successful DZ Open Ice Dekes,Failed DZ Open Ice Dekes,DZ Open Ice Deke Success Rate,Successful NZ Open Ice Dekes,Failed NZ Open Ice Dekes,NZ Open Ice Deke Success Rate,...,Total DZ Penalties,Total NZ Undisciplined Penalties,Total NZ Obstruction Penalties,Total NZ Penalties,Total Undisciplined Penalties,Total Obstruction Penalties,Total Penalties,Unnamed: 0,Conference,conference
0,Air Force Academy Falcons,203,664,0.234141,99,277,0.263298,56,210,0.210526,...,65,14,7,21,67,71,138,0,Atlantic Hockey,Atlantic Hockey
1,Arizona State University Sun Devils,138,531,0.206278,93,275,0.252717,53,203,0.207031,...,58,15,10,25,58,77,135,1,NCHC,NCHC
2,Army Black Knights,144,551,0.207194,66,264,0.200000,62,220,0.219858,...,71,8,9,17,62,81,143,2,Atlantic Hockey,Atlantic Hockey
3,Augustana University Vikings,188,653,0.223543,114,390,0.226190,81,257,0.239645,...,58,9,8,17,49,64,113,3,CCHA,CCHA
4,Bemidji State University Beavers,182,573,0.241060,76,228,0.250000,63,194,0.245136,...,57,13,10,23,54,70,124,4,CCHA,CCHA


In [25]:
scaler = StandardScaler()
scaled_df = scaler.fit_transform(numeric_metrics)

df_scaled = pd.DataFrame(scaled_df, columns=numeric_metrics.columns)
df_scaled.insert(0, 'Team', teams)
df_scaled.insert(1, 'Conference', conferences)
print(f'Dataset Shape: {df_scaled.shape}')
df_scaled.head()

Dataset Shape: (63, 389)


,Team,Conference,Successful OZ Open Ice Dekes,Failed OZ Open Ice Dekes,OZ Open Ice Deke Success Rate,Successful DZ Open Ice Dekes,Failed DZ Open Ice Dekes,DZ Open Ice Deke Success Rate,Successful NZ Open Ice Dekes,Failed NZ Open Ice Dekes,...,Total DZ Undisciplined Penalties,Total DZ Obstruction Penalties,Total DZ Penalties,Total NZ Undisciplined Penalties,Total NZ Obstruction Penalties,Total NZ Penalties,Total Undisciplined Penalties,Total Obstruction Penalties,Total Penalties,Unnamed: 0
0,Air Force Academy Falcons,Atlantic Hockey,-0.162420,0.359118,-0.480970,0.134802,-0.254371,0.454495,-1.069156,-0.681307,...,-0.057556,1.124145,0.622713,0.257343,-1.192069,-0.627193,-0.007824,0.544667,0.281587,-1.704773
1,Arizona State University Sun Devils,NCHC,-1.680934,-1.290582,-1.510774,-0.145171,-0.309252,0.107220,-1.267653,-0.918260,...,-0.807775,0.953492,-0.088729,0.527552,-0.381633,0.097819,-0.641554,1.172363,0.115793,-1.649780
2,Army Black Knights,Atlantic Hockey,-1.540764,-1.042507,-1.476911,-1.405048,-0.611099,-1.623082,-0.672161,-0.342803,...,0.067480,1.977412,1.232521,-1.363915,-0.651778,-1.352206,-0.359896,1.590827,0.557910,-1.594787
3,Augustana University Vikings,CCHA,-0.512846,0.222677,-0.872647,0.834733,2.846425,-0.763452,0.584990,0.909663,...,-0.807775,0.953492,-0.088729,-1.093706,-0.921924,-1.352206,-1.275285,-0.187644,-1.100030,-1.539795
4,Bemidji State University Beavers,CCHA,-0.653017,-0.769624,-0.225249,-0.938427,-1.598963,0.018029,-0.605995,-1.222913,...,-1.307921,1.465452,-0.190363,-0.012867,-0.381633,-0.264687,-0.923212,0.440051,-0.492119,-1.484802


**Now with our teams properly seperated by conference, we are able to compare and contrast conferences and perform EDA**

# EDA on Conferece Play

In [26]:
conference_baselines = df_scaled.groupby('Conference').mean(numeric_only=True)

conference_baselines.head()

,Successful OZ Open Ice Dekes,Failed OZ Open Ice Dekes,OZ Open Ice Deke Success Rate,Successful DZ Open Ice Dekes,Failed DZ Open Ice Dekes,DZ Open Ice Deke Success Rate,Successful NZ Open Ice Dekes,Failed NZ Open Ice Dekes,NZ Open Ice Deke Success Rate,Successful Open Ice Dekes,...,Total DZ Undisciplined Penalties,Total DZ Obstruction Penalties,Total DZ Penalties,Total NZ Undisciplined Penalties,Total NZ Obstruction Penalties,Total NZ Penalties,Total Undisciplined Penalties,Total Obstruction Penalties,Total Penalties,Unnamed: 0
Conference,,,,,,,,,,,,,,,,,,,,,
Atlantic Hockey,0.012793,0.044063,-0.048808,0.382111,0.522200,0.120763,0.373259,0.307125,0.179055,0.216652,...,0.292546,0.714577,0.663367,-0.093930,0.347758,0.170320,0.224544,0.837592,0.618701,-0.615918
Big Ten,1.005668,0.656809,0.767877,0.302785,-0.150096,0.530402,0.664389,0.530538,0.451398,0.897750,...,0.017465,-0.582389,-0.332652,0.527552,-0.327604,0.134070,0.245668,-0.187644,0.093687,0.472937
CCHA,-0.193569,-0.239019,-0.013960,0.160725,0.401156,-0.044203,-0.253110,-0.278863,-0.122224,-0.130594,...,-0.043664,0.498416,0.261346,0.167273,0.338753,0.339490,-0.039119,-0.048157,-0.056142,-0.604919
ECAC,-0.454442,-0.198017,-0.467512,-0.347373,0.004029,-0.507456,-0.269652,-0.007119,-0.337996,-0.466706,...,-0.120075,-0.027765,-0.114137,-0.598321,-0.314097,-0.612089,-0.500725,-0.309696,-0.556594,-0.151230
Hockey East,0.202873,-0.020888,0.302585,0.020267,-0.249381,0.224372,0.013558,-0.278179,0.215398,0.140487,...,-0.353097,-0.241082,-0.430590,-0.283077,-0.455309,-0.495373,-0.244673,-0.216176,-0.306229,0.354953


In [27]:
display(df_scaled.columns)
display(df_scaled.shape)

Index(['Team', 'Conference', 'Successful OZ Open Ice Dekes',
       'Failed OZ Open Ice Dekes', 'OZ Open Ice Deke Success Rate',
       'Successful DZ Open Ice Dekes', 'Failed DZ Open Ice Dekes',
       'DZ Open Ice Deke Success Rate', 'Successful NZ Open Ice Dekes',
       'Failed NZ Open Ice Dekes',
       ...
       'Total DZ Undisciplined Penalties', 'Total DZ Obstruction Penalties',
       'Total DZ Penalties', 'Total NZ Undisciplined Penalties',
       'Total NZ Obstruction Penalties', 'Total NZ Penalties',
       'Total Undisciplined Penalties', 'Total Obstruction Penalties',
       'Total Penalties', 'Unnamed: 0'],
      dtype='object', length=389)

(63, 389)

**Have to statistically prove that the difference in conference play is significant**

In [28]:
missing_filter = df_scaled['Conference'].isna()
problem_teams = df_scaled[missing_filter]
print(problem_teams[['Team', 'Conference']])


                                       Team Conference
33      Penn State University Nittany Lions        NaN
48  University Of Notre Dame Fighting Irish        NaN


In [29]:
df_scaled.loc[[33, 48], 'Conference'] = 'Big Ten'

In [30]:
from tqdm import tqdm
import scipy.stats as stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd

tukey_loop = df_scaled.copy()

metrics_to_test = [col for col in tukey_loop.columns if col not in ['Team', 'Conference']]

all_anova_results = {}
significant_tukey_reports = {}

for metric in tqdm(metrics_to_test, desc="Analyzing Hockey Metrics"):
    groups = [group[metric].values for name, group in df_scaled.groupby('Conference')]
    f_stat, anova_p_value = stats.f_oneway(*groups)
    
    all_anova_results[metric] = {
        'F-Stat': f_stat, 
        'P-Value': anova_p_value,
        'Significant': anova_p_value <= 0.05
    }
    
    if anova_p_value <= 0.05:
        print(f"Kept: {metric} (p={anova_p_value:.4f})")
        
        tukey_results = pairwise_tukeyhsd(endog=df_scaled[metric],
                                          groups=df_scaled['Conference'],
                                          alpha=0.05)
        
        significant_tukey_reports[metric] = tukey_results
    
    else: 
        tukey_loop.drop(columns=[metric], inplace=True)
        print(f'Popped: {metric} (p={anova_p_value:.4f})')

print("\n--- Loop Complete ---")
print(f"Remaining columns in tukey_loop: {tukey_loop.columns.tolist()}")

Analyzing Hockey Metrics:   0%|          | 0/387 [00:00<?, ?it/s]

Kept: Successful OZ Open Ice Dekes (p=0.0221)


Analyzing Hockey Metrics:   0%|          | 1/387 [00:00<01:31,  4.22it/s]

Kept: Failed OZ Open Ice Dekes (p=0.0077)


Analyzing Hockey Metrics:   1%|          | 2/387 [00:00<01:32,  4.14it/s]

...

Popped: Total DZ Penalties (p=0.2440)
Kept: Total NZ Undisciplined Penalties (p=0.0127)


Analyzing Hockey Metrics:  99%|█████████▊| 382/387 [00:40<00:00,  9.54it/s]

Popped: Total NZ Obstruction Penalties (p=0.2888)
Popped: Total NZ Penalties (p=0.0533)
Popped: Total Undisciplined Penalties (p=0.0540)
Kept: Total Obstruction Penalties (p=0.0039)


Analyzing Hockey Metrics: 100%|██████████| 387/387 [00:40<00:00,  9.55it/s]

Popped: Total Penalties (p=0.0956)
Kept: Unnamed: 0 (p=0.0342)

--- Loop Complete ---
Remaining columns in tukey_loop: ['Team', 'Conference', 'Successful OZ Open Ice Dekes', 'Failed OZ Open Ice Dekes', 'Successful Open Ice Dekes', 'Failed Open Ice Dekes', 'Dump In Rate', 'Total Dump In Attempts', 'Dump Ins Below Hash Marks', 'Total Chip In Attempts', 'Chip Ins Below Hash Marks', 'Total Self Chip-In Recoveries', 'Expected Goals', 'Grade A Shot Opportunities', 'Grade C Shot Opportunities', 'Total OZ Entry Pass attempts in the NZ', 'Successful OZ Entry Pass Attempts', 'Total North Pass Attempts in the NZ', 'Successful North NZ Pass Attempts', 'Total South Pass Attempts in the NZ', 'Successful South NZ Pass Attempts', 'Total East-West Pass Attempts in the NZ', 'Successful East-West NZ Pass Attempts', 'NZ East-West Passing Success Rate', 'NZ Pass Attempts', 'Successful NZ Passes', 'Total OZ Pass Receptions', 'Total OZ Pass Receptions in the Slot', 'Total OZ Pass Receptions Outside the Slot'

In [ ]:
tukey_loop.to_csv('ss_cols_df.csv')

cleaned_df = pd.read_csv('ss_cols_df.csv')

cleaned_conference_baselines = cleaned_df.groupby('Conference').mean(numeric_only=True)



In [32]:
cleaned_conference_baselines


,Unnamed: 0.1,Successful OZ Open Ice Dekes,Failed OZ Open Ice Dekes,Successful Open Ice Dekes,Failed Open Ice Dekes,Dump In Rate,Total Dump In Attempts,Dump Ins Below Hash Marks,Total Chip In Attempts,Chip Ins Below Hash Marks,...,Successful NZ Blocked Passes,Successful Blocked Passes,DZ Reception Preventions,Total Reception Preventions,Total OZ Obstruction Penalties,Total OZ Penalties,Total DZ Obstruction Penalties,Total NZ Undisciplined Penalties,Total Obstruction Penalties,Unnamed: 0
Conference,,,,,,,,,,,,,,,,,,,,,
Atlantic Hockey,19.800000,0.012793,0.044063,0.216652,0.286144,0.463172,0.561241,0.514828,0.475375,0.448341,...,-0.021864,0.042433,0.145471,-0.131051,0.449798,0.409561,0.714577,-0.093930,0.837592,-0.615918
Big Ten,39.857143,0.962282,0.733003,0.836184,0.619693,-1.301132,-0.487698,-0.471846,-0.752269,-0.755214,...,0.407089,0.576650,0.487216,0.635648,0.259444,0.721831,-0.314219,0.643356,-0.008303,0.487078
CCHA,20.000000,-0.193569,-0.239019,-0.130594,-0.116506,0.707995,0.732614,0.682087,0.609535,0.593783,...,0.478901,0.670884,0.962295,0.916047,-0.823454,-0.599782,0.498416,0.167273,-0.048157,-0.604919
ECAC,28.250000,-0.454442,-0.198017,-0.466706,-0.144880,0.001229,-0.489008,-0.505586,-0.399811,-0.408798,...,-0.662549,-0.759222,-0.679242,-0.553965,-0.290465,-0.611987,-0.027765,-0.598321,-0.309696,-0.151230
Hockey East,37.454545,0.202873,-0.020888,0.140487,-0.171655,-0.204844,-0.211968,-0.122970,-0.048272,-0.017731,...,-0.111981,-0.107158,-0.085905,-0.232441,0.183303,0.156920,-0.241082,-0.283077,-0.216176,0.354953
Independent,36.600000,-0.867945,-1.278178,-0.982803,-1.556967,0.498194,-0.717030,-0.737938,-0.460608,-0.398639,...,-0.632772,-1.373303,-0.826707,-0.857115,-0.669479,0.112982,-1.094349,1.176055,-1.191958,0.307959
NCHC,40.222222,0.271071,0.619598,0.236074,0.584541,-0.238689,0.332539,0.347250,0.295342,0.283651,...,0.600570,0.739676,-0.032943,0.234061,0.657071,0.144715,-0.108351,-0.072914,0.463299,0.507155


In [33]:
cleaned_conference_baselines.to_csv('cleaned_conference_baselines.csv')